# Day 07 - 聚类：给天气分分组今天我们来学习 **无监督学习** —— 聚类 (Clustering)。和监督学习不同，聚类不需要 "答案"。它会自动把相似的数据分到一组。---## 今天的目标1. 理解无监督学习2. 用 K-Means 给城市按气候分组3. 可视化聚类结果

## 1. 什么是聚类？想象你有一堆不同颜色的弹珠混在一起，你想把它们按颜色分开。但你不知道有哪些颜色，也不知道每种颜色叫什么名字。**聚类** 就是让计算机自动把相似的弹珠（数据）分到一组。今天我们用 **K-Means** 算法，把中国30个城市按气候特征分成4组。

In [ ]:
# 导入库import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.cluster import KMeansfrom sklearn.preprocessing import StandardScalerplt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]plt.rcParams["axes.unicode_minus"] = False# 读取数据df = pd.read_csv("data/weather_sample.csv", encoding="utf-8-sig")print(f"数据大小: {df.shape}")

In [ ]:
# 计算每个城市的气候特征city_features = df.groupby("城市").agg({    "温度": "mean",    "湿度": "mean",    "风速": "mean",    "气压": "mean",    "降雨量": "mean",    "是否下雨": "mean",}).round(2)print("各城市气候特征:")city_features.head()

## 2. 数据标准化不同特征的数值范围差别很大（温度0-40，湿度20-99，气压990-1030）。**标准化 (Standardization)** 把所有特征变成相同的范围，这样每个特征对聚类的影响是一样的。公式：`标准化值 = (原始值 - 平均值) / 标准差`

In [ ]:
# 标准化数据scaler = StandardScaler()X_scaled = scaler.fit_transform(city_features)print("标准化前:")print(city_features.head().to_string())print(f"\n标准化后 (前5个城市):")print(pd.DataFrame(X_scaled, columns=city_features.columns, index=city_features.index).head().to_string())

## 3. K-Means 聚类K-Means 的做法：1. 随机选 K 个中心点2. 把每个数据点分到最近的中心3. 更新中心点的位置4. 重复 2-3 直到中心不再变化我们选 K=4，把城市分成4个气候区。

In [ ]:
# 训练 K-Means 模型kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)city_features["气候区"] = kmeans.fit_predict(X_scaled)# 看看每个气候区有哪些城市for i in range(4):    group = city_features[city_features["气候区"] == i].index.tolist()    print(f"气候区 {i}: {group}")

In [ ]:
# 每个气候区的平均特征group_stats = city_features.groupby("气候区").mean().round(2)print("\n各气候区特征:")print(group_stats.to_string())

In [ ]:
# 可视化：用温度和湿度两个维度plt.figure(figsize=(12, 8))colors = ["red", "blue", "green", "orange"]markers = ["o", "s", "^", "D"]for i in range(4):    group = city_features[city_features["气候区"] == i]    plt.scatter(group["温度"], group["湿度"],                c=colors[i], marker=markers[i], s=100,                label=f"气候区 {i}", alpha=0.7, edgecolors="black")    # 标注城市名    for city in group.index:        plt.annotate(city, (group.loc[city, "温度"], group.loc[city, "湿度"]),                     fontsize=9, ha="center", va="bottom")plt.xlabel("平均温度 (C)", fontsize=14)plt.ylabel("平均湿度 (%)", fontsize=14)plt.title("城市气候聚类结果", fontsize=16)plt.legend(fontsize=12)plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()

In [ ]:
# 肘部法则：K 选多少合适？inertias = []K_range = range(2, 10)for k in K_range:    km = KMeans(n_clusters=k, random_state=42, n_init=10)    km.fit(X_scaled)    inertias.append(km.inertia_)plt.figure(figsize=(8, 5))plt.plot(K_range, inertias, "bo-", linewidth=2)plt.xlabel("K (聚类数)", fontsize=14)plt.ylabel("惯性 (Inertia)", fontsize=14)plt.title("肘部法则：选择最佳 K 值", fontsize=16)plt.grid(True, alpha=0.3)plt.tight_layout()plt.show()

---## 你学到了什么？| 概念 | 说明 ||------|------|| 无监督学习 | 不需要答案的学习 || 聚类 | 自动把相似数据分组 || K-Means | 最常用的聚类算法 || 标准化 | 让不同特征在同一尺度 || 肘部法则 | 选择最佳 K 值的方法 |---## 挑战任务（选做）1. 试试 K=3 或 K=5，分组结果有什么不同？2. 用湿度和降雨量做聚类，看看结果3. 查一下你所在的城市属于哪个气候区